#### Simple Gen AI APP Using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

2026-04-21 19:04:35.708704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776794675.771457   82356 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776794675.794357   82356 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-21 19:04:35.934126: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# loader=WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
loader=WebBaseLoader("https://www.radius.com/en-gb/")
loader

In [4]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content="Radius | Fleet Mobility and Connectivity Solutions\n\n\n\n\n\n\n\n\nLoginHomeOur solutionsOur solutionsFuel cardsFuel cardsTelematicsTelematicsTelematicsVehicle trackingAsset trackingVehicle camerasKnowledge hubLoginInsuranceInsuranceVehicle insuranceBusiness insuranceClaims managementVehicle solutionsVehicle solutionsVehicle leasingVehicle hireSalary sacrificeNew business vehicle financeTelecomsTelecomsBusiness mobilesSIM onlyCloud communicationsBusiness internetIT and securityEV chargingEV chargingEV charge cardsEV charging pointsEV charging softwareEnergyEnergyExpense managementExpense managementPartnershipsOur missionOur officesCareersLeadership teamNewsSustai

In [5]:
len(docs)

1

In [6]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [12]:
documents

[Document(metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content='Radius | Fleet Mobility and Connectivity Solutions'),
 Document(metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content='LoginHomeOur solutionsOur solutionsFuel cardsFuel cardsTelematicsTelematicsTelematicsVehicle trackingAsset trackingVehicle camerasKnowledge hubLoginInsuranceInsuranceVehicle insuranceBusiness insuranceClaims managementVehicle solutionsVehicle solutionsVehicle leasingVehicle hireSalary sacr

In [7]:
len(documents)

9

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [9]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [10]:
vectorstoredb

In [11]:
## Query From a vector db
# query="LangSmith has two usage limits: total traces and extended"
query="Industry-leading software and technology"
result=vectorstoredb.similarity_search(query)
result[0].page_content

"As industry experts, we also empower businesses to look to the future with our range of EV vehicles, charge points and energy solutions.Industry-leading software and technologyWe're constantly introducing innovative ways to keep our customers moving forward. This means not only offering products so you can improve efficiency but also providing monitoring solutions so you can enjoy an end-to-end service.\nOur data-driven, application-based solutions give customers hassle-free ways to quickly see what they need from wherever they are.\nThe Radius Velocity online portal allows you to view your fuel spend, track your vehicles and manage all your Radius products in one place.Industries we helpCompany informationOur missionFind out more about the history of Radius as well as our future plans for growth.More detailsNewsCatch up on our latest company news and updates.More detailsCareersDo you want to join the successful team at Radius? Find out more about our current vacancies.See opportuniti

In [13]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o-mini")

In [14]:
## Retrieval Chain, Document chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>

Question: {input}
"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7de471e7fad0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7de47189da90>, root_client=<openai.OpenAI object at 0x7de47189cf20>, root_async_client=<openai.AsyncOpenAI object at 0x7de471e7f830>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputPa

In [16]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"how many employees does Radius have?",
    "context":[Document(page_content="Radius has 5000 employees and is a technology company")]
})

'Radius has 5000 employees.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [17]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [18]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)


In [19]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7de471e7f800>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\nQuestion: {input}\n'), additio

In [25]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"How many employees working in radius?", "context": "radius is a technology company with 5000 employees"})

In [27]:

response["answer"]

'Radius has 3,000 team members.'

In [28]:
response['context']

[Document(id='73091007-7f6e-4911-8783-6de1eaa6ee6d', metadata={'source': 'https://www.radius.com/en-gb/', 'title': 'Radius | Fleet Mobility and Connectivity Solutions', 'description': 'Radius deliver best-in-class sustainable mobility, connectivity and technology solutions and are trusted by customers in every corner of the globe.', 'language': 'en-GB'}, page_content="Established in 1990, we're trusted by over 470,000 businesses all over the world to help them grow and become more productive.Our trusted partnersWe work with leading global partners to deliver the right solutions and services for your business.At a global scaleOur international presence means we have extensive local knowledge which currently helps over 470,000 businesses across five continents. Our dedicated team of experts are on-hand to provide support across our range of services.470,000+Customers worldwide50Offices globally3,000Team membersWhy businesses choose RadiusAbout usAs a B2B sustainable technology specialist